In [7]:
import pandas as pd
import glob
import json
import os

sample_path = f'{os.getcwd()}/../workspace/samples/'

experiments = {
    'Original': ['x86-gemini-pro', 'arm-gemini-pro'],
    'No Query Tools': ['x86-no-query-tools', 'arm-no-query-tools'],
    'No KRepair': ['x86_64-krepair-ablation', 'arm-krepair-ablation'],
}

def get_rows(path, experiment):

    summaries = sorted(glob.glob(f'{path}/*/agent_repair/summary.json'))

    rows = []
    for sample, path in enumerate(summaries):

        with open(path, 'r') as f:
            data = json.load(f)

        rows.append({
            'experiment': experiment,
            'sample': sample,
            'arch': data['summary']['arch'],
            'success': any(attempt['summary']['boot_succeeded'] == 'yes' for attempt in data['attempts']),
            'attempts': max(attempt['attempt'] for attempt in data['attempts']),
            'input_tokens': data['llm_token_usage']['input_tokens'],
            'output_tokens': data['llm_token_usage']['output_tokens'],
            'total_tokens': data['llm_token_usage']['total_tokens'],
        })

    return rows

rows = [row for experiment, folders in experiments.items() for folder in folders for row in get_rows(f'{sample_path}/{folder}', experiment)]
df = pd.DataFrame(rows)

print(df.groupby(['experiment', 'arch'])['sample'].nunique())

df.head()

experiment      arch  
No KRepair      arm64     50
                x86_64    50
No Query Tools  arm64     50
                x86_64    50
Original        arm64     50
                x86_64    50
Name: sample, dtype: int64


,experiment,sample,arch,success,attempts,input_tokens,output_tokens,total_tokens
0,Original,0,x86_64,True,16,3061379,102456,3163835
1,Original,1,x86_64,False,20,5539100,229761,5768861
2,Original,2,x86_64,True,8,1253777,51165,1304942
3,Original,3,x86_64,True,6,628640,47936,676576
4,Original,4,x86_64,True,20,5932179,223063,6155242


In [8]:
for experiment in experiments:

    samples = df[df['experiment'] == experiment]

    print(experiment)
    print(f'Samples: {len(samples)}')
    print(f'Success Rate: {samples["success"].mean():.2%}')
    print(f'Mean Attempts: {samples["attempts"].mean():.2f}')
    print(f'Mean Input Tokens: {samples["input_tokens"].mean():,.0f}')
    print(f'Mean Output Tokens: {samples["output_tokens"].mean():,.0f}')
    print(f'Mean Total Tokens: {samples["total_tokens"].mean():,.0f}')
    print(f'Total Input Tokens: {samples["input_tokens"].sum():,}')
    print(f'Total Output Tokens: {samples["output_tokens"].sum():,}')
    print(f'Total Tokens: {samples["total_tokens"].sum():,}')
    print()

Original
Samples: 100
Success Rate: 92.00%
Mean Attempts: 9.69
Mean Input Tokens: 1,778,566
Mean Output Tokens: 71,335
Mean Total Tokens: 1,849,901
Total Input Tokens: 177,856,613
Total Output Tokens: 7,133,532
Total Tokens: 184,990,145

No Query Tools
Samples: 100
Success Rate: 34.00%
Mean Attempts: 17.39
Mean Input Tokens: 142,282
Mean Output Tokens: 175,902
Mean Total Tokens: 318,183
Total Input Tokens: 14,228,151
Total Output Tokens: 17,590,185
Total Tokens: 31,818,336

No KRepair
Samples: 100
Success Rate: 89.00%
Mean Attempts: 10.20
Mean Input Tokens: 2,313,200
Mean Output Tokens: 96,608
Mean Total Tokens: 2,409,808
Total Input Tokens: 231,320,003
Total Output Tokens: 9,660,828
Total Tokens: 240,980,831

